In [3]:
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import tensorflow as tf
import numpy as np

from learnMSA.model.tf import LearnMSAModel
from learnMSA.util import SequenceDataset
from learnMSA.model.tf.training import BatchGenerator

In [4]:
model_file = "../tmp/nan_checkpoint_20260206_093107.keras"
data_file = "../../snakeMSA/data/homfam/unaligned/myb_DNA-binding.vie"

model = tf.keras.models.load_model(model_file)

Using user-specified initial model lengths: [38, 34, 38, 36]


/home/felix/miniforge3/envs/learnMSAdev2/lib/python3.12/site-packages/keras/src/saving/serialization_lib.py:749: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)


In [5]:
with SequenceDataset(data_file) as data:
    batch_gen = BatchGenerator(static_shape_mode=True)
    batch_gen.configure(data, model.context)
    seq_batch = batch_gen(np.arange(10))

In [6]:
# Does encoding work?
encoded = model.encode_batch(seq_batch)
tf.debugging.check_numerics(encoded, "Encoded batch contains NaNs or Infs")

<tf.Tensor: shape=(10, 95, 4, 24), dtype=float32, numpy=
array([[[[8.08593750e-01, 4.71115112e-03, 3.20243835e-03, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [2.26497650e-04, 9.94140625e-01, 2.13146210e-04, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [4.55322266e-02, 7.12966919e-03, 1.73339844e-02, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [1.84535980e-04, 1.53255463e-03, 3.90291214e-04, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[5.98144531e-02, 8.66699219e-03, 2.70233154e-02, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [1.09767914e-03, 1.47342682e-04, 4.05788422e-04, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [1.14517212e-02, 7.45117188e-01, 1.08566284e-02, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [7.08103180e-04, 9.49501991e-05, 2.61783600e-04, ...,
          0.00000000e+00

In [10]:
s = tf.reduce_sum(encoded, axis=-1)
tf.reduce_min(s), tf.reduce_max(s) # Should be close to 1

(<tf.Tensor: shape=(), dtype=float32, numpy=0.999739408493042>,
 <tf.Tensor: shape=(), dtype=float32, numpy=1.0002589225769043>)

In [7]:
# Does full call work?
log_likelihood = model(seq_batch)
tf.debugging.check_numerics(log_likelihood, "Encoded batch contains NaNs or Infs")

E0000 00:00:1770367890.601456   14845 check_numerics_op.cc:292] abnormal_detected_host @0x75df30400000 = {1, 0} Encoded batch contains NaNs or Infs
2026-02-06 09:51:30.601528: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: INVALID_ARGUMENT: Encoded batch contains NaNs or Infs : Tensor had NaN values


InvalidArgumentError: {{function_node __wrapped__CheckNumerics_device_/job:localhost/replica:0/task:0/device:GPU:0}} Encoded batch contains NaNs or Infs : Tensor had NaN values [Op:CheckNumerics] name: 

In [21]:
# Are the HMM parameters ok?
tf.debugging.check_numerics(
    model.phmm_layer.hmm.transitioner.explicit_transitioner.kernel,
    "HMM transition kernel contain NaNs or Infs"
)
for emitter in model.phmm_layer.hmm.emitter:
    tf.debugging.check_numerics(emitter.kernel, f"HMM emitter {emitter} kernel contain NaNs or Infs")

E0000 00:00:1770368473.492556   14845 check_numerics_op.cc:292] abnormal_detected_host @0x75df30400000 = {1, 0} HMM transition kernel contain NaNs or Infs
2026-02-06 10:01:13.492648: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: INVALID_ARGUMENT: HMM transition kernel contain NaNs or Infs : Tensor had NaN values


InvalidArgumentError: {{function_node __wrapped__CheckNumerics_device_/job:localhost/replica:0/task:0/device:GPU:0}} HMM transition kernel contain NaNs or Infs : Tensor had NaN values [Op:CheckNumerics]

In [13]:
# When the HMM fails, where does it fail?
padding = 1 - encoded[:, :, :, -1:]
forward_seq = encoded[:, :, :, :-1]

emission_scores = model.phmm_layer.hmm.emission_scores(forward_seq, padding)
tf.debugging.check_numerics(emission_scores, "Emission scores contain NaNs or Infs")

E0000 00:00:1770368219.373884   14845 check_numerics_op.cc:292] abnormal_detected_host @0x75df30400000 = {1, 0} Emission scores contain NaNs or Infs
2026-02-06 09:56:59.373972: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: INVALID_ARGUMENT: Emission scores contain NaNs or Infs : Tensor had NaN values


InvalidArgumentError: {{function_node __wrapped__CheckNumerics_device_/job:localhost/replica:0/task:0/device:GPU:0}} Emission scores contain NaNs or Infs : Tensor had NaN values [Op:CheckNumerics] name: 

In [17]:
"nans:" , tf.where(tf.math.is_nan(emission_scores)), "infs: ", tf.where(tf.math.is_inf(emission_scores))

('nans:',
 <tf.Tensor: shape=(74100, 4), dtype=int64, numpy=
 array([[ 0,  0,  0,  0],
        [ 0,  0,  0,  1],
        [ 0,  0,  0,  2],
        ...,
        [ 9, 94,  0, 75],
        [ 9, 94,  0, 76],
        [ 9, 94,  0, 77]], shape=(74100, 4))>,
 'infs: ',
 <tf.Tensor: shape=(0, 4), dtype=int64, numpy=array([], shape=(0, 4), dtype=int64)>)